In [1]:
# Imports
import sys
from pathlib import Path
BASE = Path.cwd().parent.parent
sys.path.insert(0, str(BASE)) # Go to base directory

# Numpy Pandas
import numpy as np
import pandas as pd

# Helper libraries
import re
import nltk
import clip
import string
import joblib
import requests
import chromadb
from tqdm import tqdm
from PIL import Image
from io import BytesIO
from itertools import chain
from collections import Counter
from nltk.corpus import stopwords
from clip.simple_tokenizer import SimpleTokenizer

# Torch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet18, ResNet18_Weights

# Sklearn
from sklearn.feature_extraction.text import TfidfVectorizer

# Self imports
from src.data import Data


d:\venv\mm-prod-search-and-reco-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set device to "Cuda" if available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [3]:
# Get Data

d1 = Data()
d_train = d1.get_complete_data('train')
d_val = d1.get_complete_data('validation')

# Clean data
cols = [
    "product_title",
    "product_description",
    "product_bullet_point",
    "product_brand",
    "product_color"
]

d_clean_train = d_train
d_clean_val = d_val

for col in cols:
    # Remove URLs
    d_clean_train[col] = d_clean_train[col].apply(lambda x: re.sub(r"http\S+|www\S+|https\S+","", x))
    d_clean_val[col] = d_clean_val[col].apply(lambda x: re.sub(r"http\S+|www\S+|https\S+","", x))
    # Remove HTML tags
    d_clean_train[col] = d_clean_train[col].apply(lambda x: re.sub(r"<.*?>","", x))
    d_clean_val[col] = d_clean_val[col].apply(lambda x: re.sub(r"<.*?>","", x))
    # lowercase and remove punctuation
    d_clean_train[col]  = d_clean_train[col].str.lower().str.translate(str.maketrans("", "", string.punctuation)).str.split()
    d_clean_val[col]  = d_clean_val[col].str.lower().str.translate(str.maketrans("", "", string.punctuation)).str.split()

d_clean_train["query"] = d_clean_train["query"].apply(lambda x: re.sub(r"http\S+|www\S+|https\S+","", x))
d_clean_train["query"] = d_clean_train["query"].apply(lambda x: re.sub(r"<.*?>","", x))
d_clean_train["query"]  = d_clean_train["query"].str.lower().str.translate(str.maketrans("", "", string.punctuation)).str.split()

d_clean_val["query"] = d_clean_val["query"].apply(lambda x: re.sub(r"http\S+|www\S+|https\S+","", x))
d_clean_val["query"] = d_clean_val["query"].apply(lambda x: re.sub(r"<.*?>","", x))
d_clean_val["query"]  = d_clean_val["query"].str.lower().str.translate(str.maketrans("", "", string.punctuation)).str.split()

## Get list of docs for prods and queries
d_clean_train['prod_doc'] = d_clean_train[cols].apply(lambda x: " ".join(chain.from_iterable(x)), axis= 1)
d_clean_train['query_doc'] = d_clean_train["query"].apply(lambda x: " ".join(x))

d_clean_val['prod_doc'] = d_clean_val[cols].apply(lambda x: " ".join(chain.from_iterable(x)), axis= 1)
d_clean_val['query_doc'] = d_clean_val["query"].apply(lambda x: " ".join(x))

Loading ESIC DATA...
ESIC Downloaded succesfully!
Loading SQID DATA...
SQID Downloaded succesfully!
Loading Amazon Review DATA...
Amazon Review Data Downloaded succesfully!


In [4]:
## Write MLP module to load TFIDF 10k model

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dims, activation):
        super().__init__()
        layers = []

        prev_dim = input_dim
        for dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, dim))
            layers.append(activation())
            prev_dim = dim

        layers.append(nn.Linear(prev_dim, output_dims))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [5]:
def load_tfidf_embedder():
    tfidf_input_dim = 10_000
    tfidf_output_dim = 512
    tfidf_hidden_dim = [2048]
    tfidf_activation_fn = nn.ReLU

    tfidf_model = MLP(tfidf_input_dim, tfidf_hidden_dim, tfidf_output_dim, tfidf_activation_fn)

    TFIDF_MODEL_PATH = "../../model"
    tfidf_model.load_state_dict(torch.load(f'{TFIDF_MODEL_PATH}/tfidf_10k_model.pt', weights_only=True))
    tfidf_model.eval()
    return tfidf_model

def load_clip_embedder():
    clip_model, clip_preprocess = clip.load("ViT-L/14", device=device)
    clip_model.eval()
    return clip_model, clip_preprocess

def load_resnet_embedder():
    ## Instantiate the ResNet18 model
    weights = ResNet18_Weights.DEFAULT
    resnet_model = resnet18(weights=weights)
    resnet_model.fc = nn.Identity() # Remove the last layer 512 -> 1000 (make it identity)
    resnet_model.eval()
    return resnet_model, weights


In [6]:
## Load all the models
tfidf_model = load_tfidf_embedder()
clip_model, clip_preprocess = load_clip_embedder()
resnet_model, weights = load_resnet_embedder()

In [7]:
def get_tfidf_embeddings(text, tfidf_model = tfidf_model):
    TFIDF_VECTORIZER_PATH = "../../model"
    tfidf_vectorizer = joblib.load(f"{TFIDF_VECTORIZER_PATH}/tfidf_vectorizer.pkl")
    tfidf_text = torch.tensor(tfidf_vectorizer.transform([text]).toarray().squeeze(), dtype=torch.float32)
    tfidf_feature_vector = tfidf_model(tfidf_text)
    return tfidf_feature_vector

def get_clip_text_embeddings(text, clip_model = clip_model):
    clip_tokenizer = SimpleTokenizer()
    clip_tokens =  clip_tokenizer.encode(text)
    chunks = torch.stack([torch.tensor([49406] + ([0] * (75 - len(clip_tokens[i : i + 75])) + clip_tokens[i : i + 75]) + [49407]) for i in range(0, len(clip_tokens), 75)]).to(device) ## 49406 -> SOS, 49407 -> EOS, 77-> max no. of tokens clip can take, so chose 75 (to keep space for sos and eos)

    clip_txt_embs = clip_model.encode_text(chunks)
    final_clip_emb = clip_txt_embs.mean(dim=0)
    return final_clip_emb / final_clip_emb.norm()

def get_clip_image_embeddings(image, clip_model = clip_model):
    clp_img = clip_preprocess(image).unsqueeze(0).to(device)
    clip_img_emb = clip_model.encode_image(clp_img)
    return clip_img_emb

def get_resnet_embeddings(image, resnet_model = resnet_model):
    transform = weights.transforms()
    input_img = transform(image).unsqueeze(0)
    resnet_embeddings = resnet_model(input_img)
    return resnet_embeddings
    
def get_image_from_url(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content))
        return image
    except (requests.RequestException, OSError) as e:
        return None

In [95]:
NUM_DATA = 10

def get_emb_lists(df, txt_col_name = None, img_col_name = None):
    tfidf_emb_list = []
    clip_text_emb_list = []
    clip_img_emb_list = []
    resnet_emb_list = []

    for i in tqdm(range(min(len(df), NUM_DATA))):
        with torch.no_grad():
            ## TEXT EMBEDDINGS
            if txt_col_name:
                txt = df[txt_col_name].iloc[i]
                tfidf_emb_list.append(get_tfidf_embeddings(txt).numpy())
                clip_text_emb_list.append(get_clip_text_embeddings(txt).numpy())

            ## IMAGE EMBEDDINGS
            if img_col_name:
                img_url = df[img_col_name].iloc[i]
                img = get_image_from_url(img_url)
                if img == None:
                    clip_img_emb_list.append(None)
                    resnet_emb_list.append(None)
                else:
                    clip_img_emb_list.append(get_clip_image_embeddings(img).numpy().squeeze(0))
                    resnet_emb_list.append(get_resnet_embeddings(img).numpy().squeeze(0))

    return tfidf_emb_list, clip_text_emb_list, clip_img_emb_list, resnet_emb_list

In [155]:
prod_train_df = d_clean_train[['product_id', 'prod_doc', 'image_url']].drop_duplicates(subset='product_id').reset_index(drop=True)

prod_tfidf_emb_list, prod_clip_text_emb_list, prod_clip_img_emb_list, prod_resnet_emb_list  = get_emb_lists(prod_train_df, 'prod_doc', 'image_url')

100%|██████████| 10/10 [00:16<00:00,  1.70s/it]


In [156]:
prod_train_df_with_emb = prod_train_df[:NUM_DATA].copy()
prod_train_df_with_emb['Product_TFIDF_Emb'] = prod_tfidf_emb_list
prod_train_df_with_emb['Product_CLIP_text_Emb'] = prod_clip_text_emb_list
prod_train_df_with_emb['Product_CLIP_img_Emb'] = prod_clip_img_emb_list
prod_train_df_with_emb['Product_Resnet18_Emb'] = prod_resnet_emb_list

In [ ]:
query_train_df = d_clean_train[['query_id', 'query_doc']].drop_duplicates(subset='query_id').reset_index(drop=True)

query_tfidf_emb_list, query_clip_text_emb_list, query_clip_img_emb_list, query_resnet_emb_list  = get_emb_lists(query_train_df, 'query_doc')

100%|██████████| 10/10 [00:02<00:00,  4.10it/s]


In [ ]:
query_df_with_emb = query_train_df[:NUM_DATA].copy()
query_df_with_emb['Query_TFIDF_Emb'] = query_tfidf_emb_list
query_df_with_emb['Query_CLIP_text_Emb'] = prod_clip_text_emb_list

In [160]:
train_data_with_prod_emb = pd.merge(d_clean_train, prod_train_df_with_emb, on='product_id', how='inner')
complete_train_data = pd.merge(train_data_with_prod_emb, query_df_with_emb, on='query_id', how='inner')

In [ ]:
e_rows = (complete_train_data[complete_train_data['esci_label'].eq('E')].drop_duplicates('query_id', keep='first').set_index('query_id'))
complete_train_data['Query_CLIP_img_Emb'] = (complete_train_data['query_id'].map(e_rows['Product_CLIP_img_Emb']))
complete_train_data['Query_Resnet18_Emb'] = (complete_train_data['query_id'].map(e_rows['Product_Resnet18_Emb']))

In [140]:
TRAIN_DATA_PATH = "../../data"
joblib.dump(complete_train_data, f"{TRAIN_DATA_PATH}/train_data_with_all_embeddings.pkl")

['../../data/train_data_with_all_embeddings.pkl']

In [141]:
prod_val_df = d_clean_val[['product_id', 'prod_doc', 'image_url']].drop_duplicates(subset='product_id').reset_index(drop=True)

prod_tfidf_emb_list, prod_clip_text_emb_list, prod_clip_img_emb_list, prod_resnet_emb_list  = get_emb_lists(prod_val_df, 'prod_doc', 'image_url')

100%|██████████| 10/10 [00:14<00:00,  1.46s/it]


In [142]:
prod_val_df_with_emb = prod_val_df[:NUM_DATA].copy()
prod_val_df_with_emb['Product_TFIDF_Emb'] = prod_tfidf_emb_list
prod_val_df_with_emb['Product_CLIP_text_Emb'] = prod_clip_text_emb_list
prod_val_df_with_emb['Product_CLIP_img_Emb'] = prod_clip_img_emb_list
prod_val_df_with_emb['Product_Resnet18_Emb'] = prod_resnet_emb_list

In [143]:
query_val_df = d_clean_val[['query_id', 'query_doc']].drop_duplicates(subset='query_id').reset_index(drop=True)

query_tfidf_emb_list, query_clip_text_emb_list, query_clip_img_emb_list, query_resnet_emb_list  = get_emb_lists(query_val_df, 'query_doc')

100%|██████████| 10/10 [00:02<00:00,  4.23it/s]


In [144]:
query_df_with_emb = query_val_df[:NUM_DATA].copy()
query_df_with_emb['Query_TFIDF_Emb'] = query_tfidf_emb_list
query_df_with_emb['Query_CLIP_text_Emb'] = prod_clip_text_emb_list

In [145]:
val_data_with_prod_emb = pd.merge(d_clean_val, prod_val_df_with_emb, on='product_id', how='inner')
complete_val_data = pd.merge(val_data_with_prod_emb, query_df_with_emb, on='query_id', how='inner')

In [146]:
e_rows = (complete_val_data[complete_val_data['esci_label'].eq('E')].drop_duplicates('query_id', keep='first').set_index('query_id'))
complete_val_data['Query_CLIP_img_Emb'] = (complete_val_data['query_id'].map(e_rows['Product_CLIP_img_Emb']))
complete_val_data['Query_Resnet18_Emb'] = (complete_val_data['query_id'].map(e_rows['Product_Resnet18_Emb']))

In [147]:
VAL_DATA_PATH = "../../data"
joblib.dump(complete_val_data, f"{VAL_DATA_PATH}/val_data_with_all_embeddings.pkl")

['../../data/val_data_with_all_embeddings.pkl']